# NumCompute-Stream quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dax555/Assignment_2.2_Programming_DakshShah_a3231005/blob/Individual/demo/quickstart.ipynb)

A decision-tree-based streaming learning framework built with **NumPy and matplotlib only**.
This notebook follows the four things the demo has to show:

| Step | What happens | Module |
|---|---|---|
| 1 | Load a dataset from a CSV file | `io.py` |
| 2 | Split it into chunks to simulate a stream | `stream.iter_chunks` |
| 3 | Train a pipeline incrementally with `.partial_fit()` on each chunk | `pipeline.py`, `preprocessing.py`, `tree.py`, `stream.py` |
| 4 | Log and visualise metrics over time | `metrics.py`, `stats.py`, `visualise.py` |

### How to run

**Google Colab (recommended):** open the notebook in Colab (badge above, or *File > Open notebook > GitHub*),
then *Runtime > Run all*. The setup cell clones the public repository from GitHub into the Colab machine
and imports the package from it. No GitHub token is needed.

**Locally:** open this notebook from the `demo/` folder (or the repository root) and *Restart & Run All*.
The setup cell sees the local checkout and skips the clone.

### Data: your CSV or the built-in Iris data
Section 1 loads a CSV if you give it one, and otherwise falls back to the Iris dataset (150 flowers,
4 measurements, 3 species) that is **hard-coded in this notebook**, so the notebook always runs.
Your own CSV needs a header row, numeric feature columns, and the class label in the **last** column.

## 0. Setup: get the code from GitHub (Colab) or use the local checkout

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/Dax555/Assignment_2.2_Programming_DakshShah_a3231005.git"
BRANCH = "Individual"            # the branch that holds the Assignment 2.2 code
REPO_DIR = "Assignment_2.2_Programming_DakshShah_a3231005"

try:
    import google.colab              # noqa: F401  (only importable inside Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 1) A local checkout: this notebook was opened from the repository root or its demo/ folder.
here = os.path.abspath(os.getcwd())
ROOT = next((p for p in (here, os.path.dirname(here), os.path.join(here, REPO_DIR))
             if os.path.isdir(os.path.join(p, "numcompute_stream"))), None)

# 2) Otherwise (e.g. on Colab) clone the repository from GitHub.
if ROOT is None:
    ROOT = os.path.abspath(REPO_DIR)
    print(f"cloning {REPO_URL} (branch {BRANCH}) ...")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

%matplotlib inline
import numpy as np
import matplotlib

import numcompute_stream as ncs
from numcompute_stream import io, metrics, stats, visualise
from numcompute_stream.pipeline import Pipeline
from numcompute_stream.preprocessing import Imputer, StandardScaler
from numcompute_stream.stream import StreamTrainer, iter_chunks
from numcompute_stream.tree import DecisionTreeClassifier

print("running on", "Google Colab" if IN_COLAB else "a local machine")
print("numcompute_stream", ncs.__version__, "| numpy", np.__version__,
      "| matplotlib", matplotlib.__version__)
print("package loaded from:", os.path.dirname(ncs.__file__))

## 1. Load the dataset with `io.py`

Pick the data source in the next cell:

- **Upload a CSV (Colab):** set `UPLOAD_CSV = True`; a file picker appears when the cell runs.
- **A CSV already on disk:** set `CSV_PATH` to its path (on Colab you can also drag a file into the *Files* panel).
- **Neither:** leave the defaults. If no CSV is found, the hard-coded Iris data below is used.

`load_mixed_csv` returns the numeric columns as a float array and the text columns as a string array,
with the rows lined up. The last column is taken as the class label, every other column as a feature.

In [ ]:
UPLOAD_CSV = False   # True -> on Colab, pick a CSV from your computer
CSV_PATH = ""        # or a path to a CSV file, e.g. "my_data.csv"

# Built-in fallback: the Iris dataset (header + 150 rows), used when no CSV is given or found.
IRIS_CSV = """\
sepal_length,sepal_width,petal_length,petal_width,species
5.1,3.5,1.4,0.2,setosa
4.9,3.0,1.4,0.2,setosa
4.7,3.2,1.3,0.2,setosa
4.6,3.1,1.5,0.2,setosa
5.0,3.6,1.4,0.2,setosa
5.4,3.9,1.7,0.4,setosa
4.6,3.4,1.4,0.3,setosa
5.0,3.4,1.5,0.2,setosa
4.4,2.9,1.4,0.2,setosa
4.9,3.1,1.5,0.1,setosa
5.4,3.7,1.5,0.2,setosa
4.8,3.4,1.6,0.2,setosa
4.8,3.0,1.4,0.1,setosa
4.3,3.0,1.1,0.1,setosa
5.8,4.0,1.2,0.2,setosa
5.7,4.4,1.5,0.4,setosa
5.4,3.9,1.3,0.4,setosa
5.1,3.5,1.4,0.3,setosa
5.7,3.8,1.7,0.3,setosa
5.1,3.8,1.5,0.3,setosa
5.4,3.4,1.7,0.2,setosa
5.1,3.7,1.5,0.4,setosa
4.6,3.6,1.0,0.2,setosa
5.1,3.3,1.7,0.5,setosa
4.8,3.4,1.9,0.2,setosa
5.0,3.0,1.6,0.2,setosa
5.0,3.4,1.6,0.4,setosa
5.2,3.5,1.5,0.2,setosa
5.2,3.4,1.4,0.2,setosa
4.7,3.2,1.6,0.2,setosa
4.8,3.1,1.6,0.2,setosa
5.4,3.4,1.5,0.4,setosa
5.2,4.1,1.5,0.1,setosa
5.5,4.2,1.4,0.2,setosa
4.9,3.1,1.5,0.2,setosa
5.0,3.2,1.2,0.2,setosa
5.5,3.5,1.3,0.2,setosa
4.9,3.6,1.4,0.1,setosa
4.4,3.0,1.3,0.2,setosa
5.1,3.4,1.5,0.2,setosa
5.0,3.5,1.3,0.3,setosa
4.5,2.3,1.3,0.3,setosa
4.4,3.2,1.3,0.2,setosa
5.0,3.5,1.6,0.6,setosa
5.1,3.8,1.9,0.4,setosa
4.8,3.0,1.4,0.3,setosa
5.1,3.8,1.6,0.2,setosa
4.6,3.2,1.4,0.2,setosa
5.3,3.7,1.5,0.2,setosa
5.0,3.3,1.4,0.2,setosa
7.0,3.2,4.7,1.4,versicolor
6.4,3.2,4.5,1.5,versicolor
6.9,3.1,4.9,1.5,versicolor
5.5,2.3,4.0,1.3,versicolor
6.5,2.8,4.6,1.5,versicolor
5.7,2.8,4.5,1.3,versicolor
6.3,3.3,4.7,1.6,versicolor
4.9,2.4,3.3,1.0,versicolor
6.6,2.9,4.6,1.3,versicolor
5.2,2.7,3.9,1.4,versicolor
5.0,2.0,3.5,1.0,versicolor
5.9,3.0,4.2,1.5,versicolor
6.0,2.2,4.0,1.0,versicolor
6.1,2.9,4.7,1.4,versicolor
5.6,2.9,3.6,1.3,versicolor
6.7,3.1,4.4,1.4,versicolor
5.6,3.0,4.5,1.5,versicolor
5.8,2.7,4.1,1.0,versicolor
6.2,2.2,4.5,1.5,versicolor
5.6,2.5,3.9,1.1,versicolor
5.9,3.2,4.8,1.8,versicolor
6.1,2.8,4.0,1.3,versicolor
6.3,2.5,4.9,1.5,versicolor
6.1,2.8,4.7,1.2,versicolor
6.4,2.9,4.3,1.3,versicolor
6.6,3.0,4.4,1.4,versicolor
6.8,2.8,4.8,1.4,versicolor
6.7,3.0,5.0,1.7,versicolor
6.0,2.9,4.5,1.5,versicolor
5.7,2.6,3.5,1.0,versicolor
5.5,2.4,3.8,1.1,versicolor
5.5,2.4,3.7,1.0,versicolor
5.8,2.7,3.9,1.2,versicolor
6.0,2.7,5.1,1.6,versicolor
5.4,3.0,4.5,1.5,versicolor
6.0,3.4,4.5,1.6,versicolor
6.7,3.1,4.7,1.5,versicolor
6.3,2.3,4.4,1.3,versicolor
5.6,3.0,4.1,1.3,versicolor
5.5,2.5,4.0,1.3,versicolor
5.5,2.6,4.4,1.2,versicolor
6.1,3.0,4.6,1.4,versicolor
5.8,2.6,4.0,1.2,versicolor
5.0,2.3,3.3,1.0,versicolor
5.6,2.7,4.2,1.3,versicolor
5.7,3.0,4.2,1.2,versicolor
5.7,2.9,4.2,1.3,versicolor
6.2,2.9,4.3,1.3,versicolor
5.1,2.5,3.0,1.1,versicolor
5.7,2.8,4.1,1.3,versicolor
6.3,3.3,6.0,2.5,virginica
5.8,2.7,5.1,1.9,virginica
7.1,3.0,5.9,2.1,virginica
6.3,2.9,5.6,1.8,virginica
6.5,3.0,5.8,2.2,virginica
7.6,3.0,6.6,2.1,virginica
4.9,2.5,4.5,1.7,virginica
7.3,2.9,6.3,1.8,virginica
6.7,2.5,5.8,1.8,virginica
7.2,3.6,6.1,2.5,virginica
6.5,3.2,5.1,2.0,virginica
6.4,2.7,5.3,1.9,virginica
6.8,3.0,5.5,2.1,virginica
5.7,2.5,5.0,2.0,virginica
5.8,2.8,5.1,2.4,virginica
6.4,3.2,5.3,2.3,virginica
6.5,3.0,5.5,1.8,virginica
7.7,3.8,6.7,2.2,virginica
7.7,2.6,6.9,2.3,virginica
6.0,2.2,5.0,1.5,virginica
6.9,3.2,5.7,2.3,virginica
5.6,2.8,4.9,2.0,virginica
7.7,2.8,6.7,2.0,virginica
6.3,2.7,4.9,1.8,virginica
6.7,3.3,5.7,2.1,virginica
7.2,3.2,6.0,1.8,virginica
6.2,2.8,4.8,1.8,virginica
6.1,3.0,4.9,1.8,virginica
6.4,2.8,5.6,2.1,virginica
7.2,3.0,5.8,1.6,virginica
7.4,2.8,6.1,1.9,virginica
7.9,3.8,6.4,2.0,virginica
6.4,2.8,5.6,2.2,virginica
6.3,2.8,5.1,1.5,virginica
6.1,2.6,5.6,1.4,virginica
7.7,3.0,6.1,2.3,virginica
6.3,3.4,5.6,2.4,virginica
6.4,3.1,5.5,1.8,virginica
6.0,3.0,4.8,1.8,virginica
6.9,3.1,5.4,2.1,virginica
6.7,3.1,5.6,2.4,virginica
6.9,3.1,5.1,2.3,virginica
5.8,2.7,5.1,1.9,virginica
6.8,3.2,5.9,2.3,virginica
6.7,3.3,5.7,2.5,virginica
6.7,3.0,5.2,2.3,virginica
6.3,2.5,5.0,1.9,virginica
6.5,3.0,5.2,2.0,virginica
6.2,3.4,5.4,2.3,virginica
5.9,3.0,5.1,1.8,virginica
"""

if UPLOAD_CSV:
    if IN_COLAB:
        from google.colab import files
        uploaded = files.upload()                       # opens a file picker
        if uploaded:
            CSV_PATH = next(iter(uploaded))
    else:
        print("UPLOAD_CSV only works on Colab; set CSV_PATH to a local file instead.")

if CSV_PATH and os.path.isfile(CSV_PATH):
    DATA = os.path.abspath(CSV_PATH)
    print("using your CSV:", DATA)
else:
    if CSV_PATH:
        print(f"CSV not found: {CSV_PATH!r}")
    DATA = os.path.abspath("iris_builtin.csv")         # write the hard-coded data so io.py can read it
    with open(DATA, "w", encoding="utf-8") as handle:
        handle.write(IRIS_CSV)
    print("using the built-in Iris data ->", DATA)

with open(DATA, encoding="utf-8") as handle:
    n_cols = len(handle.readline().strip().split(","))
FEATURE_COLS = tuple(range(n_cols - 1))                # every column but the last is a feature

X, text, names = io.load_mixed_csv(DATA, numeric_cols=FEATURE_COLS, header=True)
y = text[:, 0]
feature_names = names[:len(FEATURE_COLS)]

print("X:", X.shape, X.dtype, "| y:", y.shape, y.dtype)
print("features:", feature_names)
print("classes :", dict(zip(*[a.tolist() for a in np.unique(y, return_counts=True)])))
print("first rows:\n", X[:3], y[:3])

## 2. Simulate a stream

Real streams are messy, so 4% of the cells are blanked (NaN) to give the `Imputer` something to do.
The file is sorted by species, so the rows are shuffled (with a seed) before being cut into chunks of 10;
otherwise the first chunks would contain a single class.

In [ ]:
SEED, CHUNK_SIZE = 0, 10
rng = np.random.default_rng(SEED)
X = X.copy()
gaps = rng.random(X.shape) < 0.04
X[gaps] = np.nan

def stream():
    """A fresh pass over the same shuffled stream."""
    return iter_chunks(X, y, chunk_size=CHUNK_SIZE, shuffle=True, seed=SEED)

sizes = [len(y_chunk) for _, y_chunk in stream()]
print(f"{int(gaps.sum())} missing cells injected")
print(f"{len(sizes)} chunks, sizes {sizes}")
X_first, y_first = next(iter(stream()))
print("first chunk labels:", y_first.tolist())

### Streaming statistics: `update_stats(X_chunk)`

Mean and variance are merged chunk by chunk (Welford / Chan), quantiles come from a reservoir sample.
After the last chunk they must agree with the batch values computed on all the data at once.

In [ ]:
state = None
for X_chunk, _ in stream():
    state = stats.update_stats(X_chunk, state)        # one call per chunk
summary = state.result()

print(f"{'feature':<14}{'count':>6}{'mean':>9}{'std':>8}{'min':>8}{'median':>8}{'max':>8}")
for j, name in enumerate(feature_names):
    print(f"{name[:13]:<14}{int(summary['count'][j]):>6}{summary['mean'][j]:>9.3f}{summary['std'][j]:>8.3f}"
          f"{summary['min'][j]:>8.2f}{summary['quantiles'][0.5][j]:>8.2f}{summary['max'][j]:>8.2f}")

assert np.allclose(summary["mean"], np.nanmean(X, axis=0))
assert np.allclose(summary["std"], np.nanstd(X, axis=0))
assert np.allclose(summary["quantiles"][0.5], np.nanmedian(X, axis=0))
print("\nstreaming statistics match the batch statistics")

## 3. Train the pipeline incrementally with `.partial_fit()`

The pipeline is `Imputer -> StandardScaler -> DecisionTreeClassifier`. One call,
`pipe.partial_fit(X_chunk, y_chunk)`, updates the imputer's means and the scaler's mean/variance,
transforms the chunk, and hands it to the tree, which **buffers** it. When the buffer holds
`buffer_size` samples the tree is retrained on the buffer and the buffer is cleared.

First by hand, to show the mechanics:

In [ ]:
pipe = Pipeline([
    ("impute", Imputer(strategy="mean")),
    ("scale", StandardScaler()),
    ("model", DecisionTreeClassifier(max_depth=4, buffer_size=30)),
])

for i, (X_chunk, y_chunk) in enumerate(stream(), start=1):
    pipe.partial_fit(X_chunk, y_chunk)
    tree = pipe["model"]
    print(f"chunk {i}: buffered={tree.buffer_count_:>2}  retrains={tree.n_retrains_}  "
          f"tree ready={tree.is_fitted}")
    if i == 4:
        break

print("\nprediction for the first chunk:", pipe.predict(X_first).tolist())
print("true labels                  :", y_first.tolist())

### The same loop with `StreamTrainer` and logging

`StreamTrainer.run` does *test-then-train*: every chunk is first scored with the current model (which has
never seen it) and only then used for training. It writes one log record per chunk: chunk accuracy,
cumulative accuracy, rolling accuracy, fit time and memory footprint.

`n/a` means there is no tree yet (the first buffer is still filling); `*` marks a retrain.
Watch the memory column: it grows while the buffer fills and drops when the buffer is cleared.

In [ ]:
def make_pipeline(buffer_size):
    return Pipeline([
        ("impute", Imputer(strategy="mean")),
        ("scale", StandardScaler()),
        ("model", DecisionTreeClassifier(max_depth=4, buffer_size=buffer_size)),
    ])

trainer = StreamTrainer(
    make_pipeline(buffer_size=30),
    metrics={"macro_f1": metrics.StreamingF1(average="macro"),
             "confusion": metrics.StreamingConfusionMatrix()},
    window=50, name="buffer_size=30", verbose=True)
history = trainer.run(stream())

## 4. Read the log and the metrics

In [ ]:
info = trainer.summary()
for key in ("chunks", "samples", "scored_chunks", "n_retrains", "cumulative_accuracy",
            "rolling_accuracy", "macro_f1", "total_fit_seconds", "peak_memory_bytes"):
    value = info[key]
    print(f"{key:<22}: {value:.4f}" if isinstance(value, float) else f"{key:<22}: {value}")

cm = trainer.metrics["confusion"]
print("\nconfusion matrix accumulated over the stream (rows = truth, columns = prediction)")
print(f"{'':<12}" + "".join(f"{c:>12}" for c in cm.labels_))
for label, row in zip(cm.labels_, cm.result()):
    print(f"{label:<12}" + "".join(f"{int(v):>12}" for v in row))

print("\ntree after the last retrain (thresholds are in scaled units):")
print(trainer.model["model"].export_text(feature_names))

### Visualise with `visualise.py`

`plot_metric_over_time` draws one metric across chunks. The first three chunks have no value because
the tree did not exist yet; they are shown as a gap rather than as zero.

In [ ]:
visualise.plot_metric_over_time(
    trainer.get_metric("cumulative_accuracy"),
    title="Cumulative accuracy over time (buffer_size=30)", ylabel="Cumulative accuracy");

In [ ]:
visualise.plot_metric_over_time(
    trainer.get_metric("error"), title="Error rate of each chunk", ylabel="Chunk error rate");

### Model comparison: two buffer sizes

`compare_models` puts two models on the same axis. A larger buffer retrains less often on more data:
it starts predicting later, but each tree has seen more examples.

In [ ]:
other = StreamTrainer(make_pipeline(buffer_size=60), window=50, name="buffer_size=60")
other.run(stream())

for run in (trainer, other):
    s = run.summary()
    print(f"{run.name:<16} retrains={s['n_retrains']}  cumulative accuracy={s['cumulative_accuracy']:.3f}  "
          f"total fit={s['total_fit_seconds'] * 1e3:.2f} ms  peak memory={s['peak_memory_bytes'] / 1024:.1f} KiB")

visualise.compare_models(
    trainer.get_metric("cumulative_accuracy"), other.get_metric("cumulative_accuracy"),
    labels=(trainer.name, other.name),
    title="Two buffer sizes on the same stream", ylabel="Cumulative accuracy");

### Predictions vs ground truth on the latest chunk

An open circle is the true class, a filled dot a correct prediction, a red cross a wrong one.

In [ ]:
print("true     :", trainer.last_y_true_.tolist())
print("predicted:", trainer.last_y_pred_.tolist())
visualise.plot_predictions_vs_ground_truth(trainer.last_y_true_, trainer.last_y_pred_);

### Saving instead of showing

Every plotting function takes `save_path=` and `show=False`, for scripts and pipeline logs.
The chunk log itself can be written to CSV.

In [ ]:
OUT = os.path.join(ROOT, "demo", "outputs")
os.makedirs(OUT, exist_ok=True)
fig, ax = visualise.plot_metric_over_time(
    trainer.get_metric("rolling_accuracy"), "Rolling accuracy (last 50 samples)", "Rolling accuracy",
    save_path=os.path.join(OUT, "notebook_rolling_accuracy.png"), show=False)
matplotlib.pyplot.close(fig)
log_file = trainer.save_log(os.path.join(OUT, "notebook_stream_log.csv"))
print("saved to", OUT, ":", sorted(f for f in os.listdir(OUT) if f.startswith("notebook_")))

## 5. Reading the file itself as a stream

The chunks above came from an array in memory. `io.iter_csv` reads the CSV a few lines at a time, so a
file larger than memory can feed the same streaming components. Here the running statistics and a
streaming histogram of the last feature (petal width for Iris) are updated straight from the file.

In [ ]:
H = len(FEATURE_COLS) - 1                                         # histogram column
lo, hi = float(np.nanmin(X[:, H])), float(np.nanmax(X[:, H]))
running = stats.RunningStats()
hist = stats.StreamingHistogram(bins=6, range=(np.floor(lo), np.ceil(hi) if hi > lo else lo + 1))
for chunk, cols in io.iter_csv(DATA, chunk_size=40, header=True, usecols=FEATURE_COLS):
    running.update_stats(chunk)
    hist.update(chunk[:, H])
    print(f"read {chunk.shape[0]:>2} rows -> mean so far {np.round(running.mean(), 3)}")

counts, edges = hist.result()
print(f"\n{feature_names[H]} histogram from the stream:")
print(stats.format_histogram(counts, edges))

## 6. Benchmarks (small versions)

The full benchmarks are in `benchmark/` (`python benchmark/stream_benchmarks.py`), with their results
saved in `benchmark/results/`. Two small ones are repeated here so their output stays in the notebook.

**(a) Loop vs vectorised.** The tree's split search scores every feature and threshold of a node. The
loop version tries them one by one; the vectorised version uses one sort and one cumulative sum.

In [ ]:
import time
sys.path.insert(0, os.path.join(ROOT, "benchmark"))
from stream_benchmarks import loop_best_split, make_classification, make_drift_stream
from numcompute_stream.tree import _best_split

def median_ms(fn, repeat=5):
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        fn()
        times.append(time.perf_counter() - start)
    return float(np.median(times)) * 1e3

print(f"{'rows':>6} {'loop (ms)':>11} {'vectorised (ms)':>16} {'speed-up':>9}  same result")
for n in (200, 1000, 2000):
    Xb, yb = make_classification(n, 8, np.random.default_rng(0))
    Xl, yl = Xb.tolist(), yb.tolist()
    a, b = loop_best_split(Xl, yl, 3), _best_split(Xb, yb, 3)
    same = a[0] == b[0] and np.isclose(a[1], b[1]) and np.isclose(a[2], b[2])
    t_loop, t_vec = median_ms(lambda: loop_best_split(Xl, yl, 3)), median_ms(lambda: _best_split(Xb, yb, 3))
    print(f"{n:>6} {t_loop:>11.2f} {t_vec:>16.3f} {t_loop / t_vec:>8.1f}x  {same}")

**(b) Buffer size.** One stream of 12,000 samples whose labelling rule changes half way (concept drift).
Small buffers retrain often and recover quickly after the change; large buffers build better trees on a
stable stream but each retrain costs more and they are slow to notice the change.

In [ ]:
Xs, ys = make_drift_stream(12000, np.random.default_rng(0))
runs = {}
print(f"{'buffer_size':>11} {'retrains':>9} {'accuracy':>9} {'before drift':>13} {'after drift':>12} "
      f"{'mean retrain (ms)':>18}")
for size in (100, 500, 2000, 4000):
    run = StreamTrainer(DecisionTreeClassifier(max_depth=6, buffer_size=size), window=1000,
                        name=f"buffer_size={size}")
    run.run(iter_chunks(Xs, ys, chunk_size=100))
    acc, fit = run.get_metric("chunk_accuracy"), run.get_metric("fit_seconds")
    retrained = np.array([r["retrained"] for r in run.history])
    runs[size] = run
    print(f"{size:>11} {run.summary()['n_retrains']:>9} {run.summary()['cumulative_accuracy']:>9.3f} "
          f"{np.nanmean(acc[:60]):>13.3f} {np.nanmean(acc[60:]):>12.3f} {fit[retrained].mean() * 1e3:>18.2f}")

visualise.compare_models(
    runs[100].get_metric("rolling_accuracy"), runs[4000].get_metric("rolling_accuracy"),
    labels=("buffer_size=100", "buffer_size=4000"),
    title="Rolling accuracy around a concept drift at chunk 60",
    ylabel="Rolling accuracy (last 1,000 samples)", xlabel="Chunk (100 samples each)");

## Summary

- `io.py` loaded the CSV (yours or the built-in Iris data), both whole and in chunks.
- `stats.update_stats`, `RunningStats`, `StreamingHistogram` kept statistics without storing the stream.
- `Pipeline.partial_fit` updated the imputer, the scaler and the buffered decision tree chunk by chunk.
- `StreamTrainer` logged per-chunk accuracy, cumulative accuracy, timing and memory.
- `visualise.py` drew the metric curves, the model comparison and predictions against ground truth.

To check the whole framework from a terminal run `python demo/iris_demo.py`, and for the tests `pytest`
(on Colab: `!cd {ROOT} && python demo/iris_demo.py`).